# LoRA & QLoRA Fine-Tuning — Class Notebook (2026-09-22)

**Course:** Full Stack Data Science with Gen AI and Agentic AI — GenAI / LLM / Fine-Tuning module

**Goal:** Fine-tune a pretrained causal language model two ways — with **LoRA** and with **QLoRA** — on a small public dataset, compare the two, and understand *why* each technique exists.

**Model:** `gpt2` (124M params, small enough to fine-tune on a laptop CPU for the LoRA part; swap in a bigger model like `mistral-7b`/`llama-3-8b` on a GPU runtime for a more realistic QLoRA demo).

**Dataset:** [`Abirate/english_quotes`](https://huggingface.co/datasets/Abirate/english_quotes) — ~2500 famous quotes with authors/tags. It's the standard tiny dataset the Hugging Face PEFT docs themselves use for LoRA demos, which makes it easy to sanity-check your results against the official example.

> **Where to run this:** the LoRA half runs fine on CPU (your Mac) in a few minutes. The QLoRA half needs an actual GPU with CUDA (bitsandbytes 4-bit quantization has no Apple Silicon / CPU backend), so run that half on **Google Colab** with a free T4 GPU runtime. Both halves use the exact same code structure so you can compare them side by side.

## 1. Concepts — what problem are we solving?

Fine-tuning a full LLM means updating *every* weight. For a 7B-parameter model that's 7 billion numbers to store gradients and optimizer state for — hundreds of GB of GPU memory. Two ideas fix this:

**LoRA (Low-Rank Adaptation).** Freeze the original weight matrix `W` entirely. Instead of learning a full update `ΔW`, learn it as a product of two tiny matrices: `ΔW = B·A`, where `A` is `(r × d)` and `B` is `(d × r)`, with rank `r` much smaller than `d` (e.g. r=8 or 16 vs d=768+). Only `A` and `B` are trained — typically <1% of the original parameter count. At inference time you can either keep the adapter separate or merge `B·A` back into `W` for zero extra latency.

**QLoRA (Quantized LoRA).** Same idea, but the frozen base model is first compressed to 4-bit precision (NF4 format) instead of the usual 16/32-bit, cutting its memory footprint by ~4x. The LoRA adapters themselves still train in higher precision (bf16), so you get near full-precision fine-tuning quality while the base model that dominates memory use sits in 4-bit. This is what makes it possible to fine-tune a 7B–70B model on a single consumer GPU.

Key hyperparameters you'll see below:

| Parameter | Meaning |
|---|---|
| `r` | Rank of the LoRA update matrices — higher = more capacity, more trainable params |
| `lora_alpha` | Scaling factor applied to the LoRA update (effective scale = alpha / r) |
| `lora_dropout` | Dropout applied inside the adapter, for regularization |
| `target_modules` | Which weight matrices get an adapter (e.g. attention query/key/value projections) |
| `bnb_4bit_quant_type` | QLoRA only — `"nf4"` is the 4-bit format from the QLoRA paper |
| `bnb_4bit_compute_dtype` | QLoRA only — precision used for the actual matmuls (bf16/fp16) |

**LoRA vs QLoRA at a glance:**

| | LoRA | QLoRA |
|---|---|---|
| Base model precision | fp16/bf16 (or fp32) | 4-bit (NF4) |
| Trainable params | Adapters only (~0.1–1%) | Adapters only (~0.1–1%) |
| GPU memory for a 7B model | ~14–28 GB | ~5–6 GB |
| Needs CUDA GPU? | No — CPU/MPS works, just slower | Yes — bitsandbytes 4-bit is CUDA-only |
| Typical use case | Small-to-mid models, any hardware | Large models on limited GPU memory |

## 2. Setup

Run this once. On Colab, uncomment the `!pip install` line.

In [ ]:
# On Google Colab, uncomment and run this first:
# !pip install -q -U transformers datasets accelerate peft bitsandbytes trl

import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel

MODEL_NAME = "gpt2"                 # swap for a bigger model (e.g. "mistralai/Mistral-7B-v0.1") when using QLoRA on a GPU
DATASET_NAME = "Abirate/english_quotes"
OUTPUT_DIR_LORA = "./gpt2-lora-quotes"
OUTPUT_DIR_QLORA = "./gpt2-qlora-quotes"

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

## 3. Load and inspect the dataset

In [ ]:
raw_dataset = load_dataset(DATASET_NAME)
print(raw_dataset)
print(raw_dataset["train"][0])

# Keep it small and fast for a class demo — 300 quotes is plenty to see the model shift style
small_dataset = raw_dataset["train"].shuffle(seed=42).select(range(300))
split_dataset = small_dataset.train_test_split(test_size=0.1, seed=42)
split_dataset

## 4. Load the base model + tokenizer, and see what it generates *before* any fine-tuning

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 has no pad token by default

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

def generate(model, prompt, max_new_tokens=40):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        top_p=0.9,
        temperature=0.8,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)

prompt = "The secret to happiness is"
print("BEFORE fine-tuning:\n", generate(base_model, prompt))

## 5. Fine-tune with LoRA

We wrap the frozen base model with small trainable adapter matrices on the attention projections. `print_trainable_parameters` will show you just how few parameters LoRA actually trains.

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["c_attn"],   # GPT-2's combined query/key/value projection
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

lora_model = get_peft_model(base_model, lora_config)
lora_model.print_trainable_parameters()

In [ ]:
def tokenize_fn(batch):
    return tokenizer(batch["quote"], truncation=True, padding="max_length", max_length=64)

tokenized_train = split_dataset["train"].map(tokenize_fn, batched=True, remove_columns=split_dataset["train"].column_names)
tokenized_eval = split_dataset["test"].map(tokenize_fn, batched=True, remove_columns=split_dataset["test"].column_names)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR_LORA,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
)

trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
)

trainer.train()

In [ ]:
lora_model.save_pretrained(OUTPUT_DIR_LORA)
tokenizer.save_pretrained(OUTPUT_DIR_LORA)

print("AFTER LoRA fine-tuning:\n", generate(lora_model, prompt))

## 6. Fine-tune with QLoRA (run this section on a GPU runtime — e.g. Colab with a T4)

Same LoRA config and same training loop as above — the only difference is *how the base model is loaded*: in 4-bit, via `BitsAndBytesConfig`. This is the part that fails on CPU/Apple Silicon, because bitsandbytes' 4-bit kernels are CUDA-only.

For a realistic demo, point `MODEL_NAME` at something bigger here (e.g. `"mistralai/Mistral-7B-v0.1"` or `"meta-llama/Llama-3-8B"`) — that's the whole point of QLoRA: it makes fine-tuning those large models possible on a single GPU.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

qlora_base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

# Prepares the 4-bit model for training (casts norm layers to fp32, enables gradient checkpointing, etc.)
qlora_base_model = prepare_model_for_kbit_training(qlora_base_model)

qlora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["c_attn"],   # for a Llama/Mistral model use ["q_proj", "k_proj", "v_proj", "o_proj"]
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

qlora_model = get_peft_model(qlora_base_model, qlora_config)
qlora_model.print_trainable_parameters()

In [ ]:
qlora_training_args = TrainingArguments(
    output_dir=OUTPUT_DIR_QLORA,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
)

qlora_trainer = Trainer(
    model=qlora_model,
    args=qlora_training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
)

qlora_trainer.train()

qlora_model.save_pretrained(OUTPUT_DIR_QLORA)
print("AFTER QLoRA fine-tuning:\n", generate(qlora_model, prompt))

## 7. Merging the adapter back into the base model (optional)

Once you're happy with an adapter, you can merge it into the base weights so downstream code doesn't need PEFT at all — useful for deployment.

In [ ]:
merged_model = lora_model.merge_and_unload()
merged_model.save_pretrained("./gpt2-lora-quotes-merged")
tokenizer.save_pretrained("./gpt2-lora-quotes-merged")
print("Merged model saved — this is a plain AutoModelForCausalLM, no PEFT wrapper needed to load it.")

## 8. Takeaways

- LoRA and QLoRA both train a tiny fraction of the total parameters by learning a low-rank update instead of the full weight matrix — that's what "parameter-efficient fine-tuning" (PEFT) means.
- QLoRA adds one extra trick on top of LoRA: load the frozen base model in 4-bit (NF4) so it takes far less memory, while the adapters still train at higher precision.
- Rule of thumb: use plain LoRA when the model already fits comfortably in memory at fp16/bf16; reach for QLoRA when it doesn't (typically 7B+ models on a single consumer/free-tier GPU).
- `target_modules` matters a lot — for GPT-2 it's `c_attn`; for Llama/Mistral-family models it's usually `q_proj`, `k_proj`, `v_proj`, `o_proj` (sometimes also `gate_proj`, `up_proj`, `down_proj`).

**Try next:**
1. Increase `r` from 8 to 32 and compare trainable-parameter counts and output quality.
2. Swap in a different small dataset (e.g. a domain-specific one relevant to your Magento background, like a set of product-description snippets) and see how the generations shift.
3. Run the QLoRA cell on Colab with `mistralai/Mistral-7B-v0.1` and compare training memory usage (`nvidia-smi`) against what plain LoRA (no quantization) would need.

**References:**
- Hugging Face PEFT docs: https://huggingface.co/docs/peft
- QLoRA paper (Dettmers et al., 2023): https://arxiv.org/abs/2305.14314
- LoRA paper (Hu et al., 2021): https://arxiv.org/abs/2106.09685